First, we need to set up the environment with the necessary Hugging Face libraries, including bitsandbytes for memory-efficient model loading and llmlingua for the base token importance.

In [1]:
!pip install -q transformers datasets accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00


We will load the TweetEval dataset, the RoBERTa classifier, and the Qwen2.5 generator.

In [2]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSequenceClassification, BitsAndBytesConfig
)
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Dataset ───────────────────────────────────────────────────────────────
print("Loading dataset...")
dataset   = load_dataset("tweet_eval", "irony")
test_data = dataset["test"]

# ── RoBERTa irony classifier ──────────────────────────────────────────────
print("Loading RoBERTa classifier...")
roberta_name  = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model     = AutoModelForSequenceClassification.from_pretrained(
    roberta_name
).to(device)

# ── Qwen 2.5-3B (4-bit) ──────────────────────────────────────────────────
print("Loading Qwen2.5-3B (4-bit quantised)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
qwen_name      = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model     = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    device_map="auto",
    quantization_config=bnb_config
)
print("All models loaded.")

Device: cuda
Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

irony/train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

irony/test-00000-of-00001.parquet:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

All models loaded.


Preprocessing

In [5]:
import torch.nn.functional as F
import nltk, re
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

STOP_WORDS = set(stopwords.words('english'))

# ── Prompt template (used by both baselines) ──────────────────────────────
STRUCTURED_PROMPT = (
    "Analyze whether the following tweet is ironic or non-ironic. "
    "Think step-by-step, then end your response with EXACTLY one of:\n"
    "Verdict: IRONIC\n"
    "Verdict: NON-IRONIC\n\n"
    "Tweet: '{tweet}'"
)


def classify_tweet(tweet_text):
    """
    Classifies the raw tweet with RoBERTa.
    Returns (predicted_label, p_ironic).
    """
    inputs = rob_tokenizer(
        tweet_text, return_tensors="pt",
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        logits = rob_model(**inputs).logits
    probs      = F.softmax(logits, dim=-1).squeeze().cpu()
    p_ironic   = probs[1].item()
    prediction = int(p_ironic >= 0.5)
    return prediction, p_ironic


def vanilla_compress(cot_text, gamma=0.50):
    """
    Trims cot_text to (1 - gamma) of its original word count.
    Drops stopwords first, then shortest remaining words.
    No gradient sensitivity, no critical-token protection.
    Returns the compressed string.
    """
    words         = cot_text.split()
    target_length = max(5, int(len(words) * (1.0 - gamma)))

    scored = []
    for idx, word in enumerate(words):
        clean = word.lower().strip(".,!?\"'")
        score = 1 if (clean in STOP_WORDS or len(clean) < 3) else 10
        scored.append((idx, word, score))

    scored.sort(key=lambda x: x[2], reverse=True)
    kept = sorted(scored[:target_length], key=lambda x: x[0])
    return " ".join(w[1] for w in kept)


def extract_structured_cot_signal(cot_text):
    """
    Extracts the verdict (IRONIC/NON-IRONIC) from the CoT text.
    Returns 1.0 for IRONIC, 0.0 for NON-IRONIC, and 0.5 if no clear verdict is found.
    """
    if "Verdict: IRONIC" in cot_text:
        return 1.0
    elif "Verdict: NON-IRONIC" in cot_text:
        return 0.0
    else:
        return 0.5 # Ambiguous or verdict not found


print("✅ Helpers ready: classify_tweet | vanilla_compress | STRUCTURED_PROMPT | extract_structured_cot_signal")

✅ Helpers ready: classify_tweet | vanilla_compress | STRUCTURED_PROMPT | extract_structured_cot_signal


Uncompressed CoT

In [6]:
import pandas as pd
from IPython.display import display

# ════════════════════════════════════════════════════════════════════════════
# BASELINE 1 — Uncompressed CoT  (γ = 0.00)
#
# What this measures:
#   • Accuracy  : Verdict extracted from Qwen's FULL, uncompressed CoT via
#                 extract_structured_cot_signal. Because the complete reasoning
#                 chain is intact, Qwen's "Verdict:" line survives and the
#                 parser returns a clean 0.0 / 1.0 signal.
#   • Avg tokens: the full word count of every generated CoT.
#
# This reflects Qwen's actual reasoning ability on irony detection — a genuine
# upper-bound CoT baseline before any compression is applied.
# ════════════════════════════════════════════════════════════════════════════

NUM_SAMPLES = 100

results_list        = []
correct_predictions = 0
total_cot_tokens    = 0

print("🚀 BASELINE 1 — Uncompressed CoT  (γ = 0.00)")
print(f"   N = {NUM_SAMPLES}\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # ── Generate full CoT ─────────────────────────────────────────────────
    prompt = STRUCTURED_PROMPT.format(tweet=sample_tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=120
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    cot_text      = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)
    cot_len       = len(cot_text.split())
    total_cot_tokens += cot_len

    # ── Extract verdict directly from Qwen's full CoT ─────────────────────
    # The uncompressed reasoning is intact, so the "Verdict:" line is present
    # and extract_structured_cot_signal returns a hard 0.0 or 1.0 signal.
    cot_signal = extract_structured_cot_signal(cot_text)
    prediction = int(cot_signal >= 0.5)   # 0.5 (ambiguous) → non-ironic (0)

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  [{i+1:02d}] CoT={cot_len:3d} words | "
          f"CoT signal={cot_signal:.1f} | Pred={prediction} | "
          f"True={true_label} | {'✅' if is_correct else '❌'}")
    print(f"        CoT snippet: {cot_text[:120].strip()!r}")

    results_list.append({
        "Tweet"           : sample_tweet[:40] + "...",
        "True"            : true_label,
        "Pred"            : prediction,
        "✓?"              : "✅" if is_correct else "❌",
        "CoT Signal"      : round(cot_signal, 1),
        "CoT Tokens"      : cot_len,
        "Tokens Removed"  : 0,
        "γ"               : 0.00,
    })

accuracy_pct   = (correct_predictions / NUM_SAMPLES) * 100
avg_cot_tokens = total_cot_tokens / NUM_SAMPLES

print(f"\n{'='*60}")
print(f"🏆 BASELINE 1 ACCURACY          : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📏 Avg CoT Tokens (uncompressed) : {avg_cot_tokens:.1f} words")
print(f"✂️  Tokens Removed               : 0  (no compression)")
print(f"🔍 Verdict source               : extract_structured_cot_signal(full_cot)")
print(f"{'='*60}")

df1 = pd.DataFrame(results_list)
display(df1)


🚀 BASELINE 1 — Uncompressed CoT  (γ = 0.00)
   N = 100

  [01] CoT= 86 words | CoT signal=1.0 | Pred=1 | True=0 | ❌
        CoT snippet: 'EXACTLY one of: Verdict: IRONIC\nVerdict: NON-IRONIC\nStep 1: Analyze the context and tone.\nThe tweet mentions a need for'
  [02] CoT= 72 words | CoT signal=1.0 | Pred=1 | True=1 | ✅
        CoT snippet: 'EXACTLY ONE RESPONSE REGARDING THE VERDICT:\n\nVerdict: IRONIC\nThe tweet uses irony by describing an action that seems ab'
  [03] CoT= 94 words | CoT signal=0.5 | Pred=1 | True=0 | ❌
        CoT snippet: "shirt. I don't even like the color. But I'm still going to wear it for the sake of the cause. \n\nStep 1: Analyze the con"
  [04] CoT= 52 words | CoT signal=0.0 | Pred=0 | True=0 | ✅
        CoT snippet: 'The tweet seems to be expressing a direct statement rather than a sarcastic or humorous observation. The use of words'
  [05] CoT= 87 words | CoT signal=0.5 | Pred=1 | True=1 | ✅
        CoT snippet: "To determine if this tweet is ironic or not,

,Tweet,True,Pred,✓?,CoT Signal,CoT Tokens,Tokens Removed,γ
0,@user Can U Help?||More conservatives ne...,0,1,❌,1.0,86,0,0.0
1,Just walked in to #Starbucks and asked f...,1,1,✅,1.0,72,0,0.0
2,#NOT GONNA WIN...,0,1,❌,0.5,94,0,0.0
3,@user He is exactly that sort of person....,0,0,✅,0.0,52,0,0.0
4,So much #sarcasm at work mate 10/10 #bor...,1,1,✅,0.5,87,0,0.0
...,...,...,...,...,...,...,...,...
95,The last day of the year and I will be t...,0,1,❌,1.0,81,0,0.0
96,MB forgot to move the elf on shelf and n...,0,1,❌,0.5,96,0,0.0
97,"@user happy new year, da! Thanks sobra f...",0,1,❌,0.5,94,0,0.0
98,WATCH: #Megatron from #Transformers Rant...,0,1,❌,1.0,92,0,0.0


Vanilla TokenSkip

In [7]:
import pandas as pd
from IPython.display import display

# ════════════════════════════════════════════════════════════════════════════
# BASELINE 2 — Vanilla TokenSkip  (γ = 0.50, static)
#
# What this measures:
#   • Accuracy  : Verdict extracted from the COMPRESSED CoT via
#                 extract_structured_cot_signal. Because vanilla_compress()
#                 prunes purely on stopword status and token length — with no
#                 task-aware gradient protection — it blindly shreds the
#                 "Verdict:" line and critical contrastive connectors (not,
#                 but, never). The parser therefore falls back to 0.5
#                 (ambiguous) or reads a corrupted conclusion, organically
#                 tanking accuracy below the uncompressed baseline.
#   • Avg tokens: compressed CoT word count (target ≈ 50% of original).
#
# No entropy, no gradients, no critical-token protection.
# This is the exact robustness failure that RDS is designed to fix.
# ════════════════════════════════════════════════════════════════════════════

VANILLA_GAMMA = 0.50
NUM_SAMPLES   = 100

results_list        = []
correct_predictions = 0
total_orig_tokens   = 0
total_comp_tokens   = 0
verdict_found_count = 0

print(f"🚀 BASELINE 2 — Vanilla TokenSkip  (γ = {VANILLA_GAMMA}, static)")
print(f"   N = {NUM_SAMPLES}\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # ── Generate CoT ──────────────────────────────────────────────────────
    prompt = STRUCTURED_PROMPT.format(tweet=sample_tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=120
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    cot_text      = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)
    orig_len      = len(cot_text.split())
    total_orig_tokens += orig_len

    # ── Static 50% compression — no critical-token protection ─────────────
    # vanilla_compress() drops stopwords and short words first, making it
    # highly likely to remove "Verdict:", "not", "but", "never", etc.
    compressed_text = vanilla_compress(cot_text, gamma=VANILLA_GAMMA)
    comp_len        = len(compressed_text.split())
    total_comp_tokens += comp_len
    tokens_removed  = orig_len - comp_len

    # ── Extract verdict from the COMPRESSED CoT ───────────────────────────
    # This is the critical difference from the buggy original: we now judge
    # the compressed reasoning — not the raw tweet — mirroring how RDS works.
    # Blind pruning will often destroy the verdict line → cot_signal = 0.5
    # (ambiguous fallback), which defaults prediction to 0 and causes errors.
    cot_signal = extract_structured_cot_signal(compressed_text)
    prediction = int(cot_signal >= 0.5)   # 0.5 (ambiguous) → non-ironic (0)

    verdict_survived = (cot_signal != 0.5)
    if verdict_survived:
        verdict_found_count += 1

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  [{i+1:02d}] Orig={orig_len:3d} → Comp={comp_len:3d} words "
          f"(removed {tokens_removed:2d}) | "
          f"CoT signal={cot_signal:.1f} {'✔verdict' if verdict_survived else '⚠️ verdict LOST'} | "
          f"Pred={prediction} | True={true_label} | {'✅' if is_correct else '❌'}")

    results_list.append({
        "Tweet"           : sample_tweet[:40] + "...",
        "True"            : true_label,
        "Pred"            : prediction,
        "✓?"              : "✅" if is_correct else "❌",
        "CoT Signal"      : round(cot_signal, 1),
        "Verdict Survived": "✔" if verdict_survived else "✘ LOST",
        "Orig Tokens"     : orig_len,
        "Comp Tokens"     : comp_len,
        "Tokens Removed"  : tokens_removed,
        "γ"               : VANILLA_GAMMA,
    })

accuracy_pct    = (correct_predictions / NUM_SAMPLES) * 100
avg_orig_tokens = total_orig_tokens / NUM_SAMPLES
avg_comp_tokens = total_comp_tokens / NUM_SAMPLES
verdict_survival_rate = (verdict_found_count / NUM_SAMPLES) * 100

print(f"\n{'='*60}")
print(f"🏆 BASELINE 2 ACCURACY                : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📏 Avg CoT Tokens — Original           : {avg_orig_tokens:.1f} words")
print(f"📏 Avg CoT Tokens — After Vanilla Skip : {avg_comp_tokens:.1f} words")
print(f"✂️  Avg Tokens Removed (static 50%)     : {avg_orig_tokens - avg_comp_tokens:.1f} words")
print(f"🔍 Verdict survived compression        : {verdict_found_count}/{NUM_SAMPLES} ({verdict_survival_rate:.0f}%)")
print(f"🔍 Verdict source                      : extract_structured_cot_signal(compressed_cot)")
print(f"{'='*60}")
print()
print("⚠️  NOTE: Low verdict survival rate above directly explains depressed accuracy.")
print("   Vanilla pruning destroys the 'Verdict:' line and contrastive connectors")
print("   ('not', 'but', 'never'), causing the parser to default to 0.5 (ambiguous)")
print("   and forcing prediction = 0 (non-ironic) for ambiguous cases.")

df2 = pd.DataFrame(results_list)
display(df2)


🚀 BASELINE 2 — Vanilla TokenSkip  (γ = 0.5, static)
   N = 100

  [01] Orig= 87 → Comp= 43 words (removed 44) | CoT signal=1.0 ✔verdict | Pred=1 | True=0 | ❌
  [02] Orig= 81 → Comp= 40 words (removed 41) | CoT signal=1.0 ✔verdict | Pred=1 | True=1 | ✅
  [03] Orig= 77 → Comp= 38 words (removed 39) | CoT signal=0.0 ✔verdict | Pred=0 | True=0 | ✅
  [04] Orig= 86 → Comp= 43 words (removed 43) | CoT signal=0.5 ⚠️ verdict LOST | Pred=1 | True=0 | ❌
  [05] Orig= 67 → Comp= 33 words (removed 34) | CoT signal=1.0 ✔verdict | Pred=1 | True=1 | ✅
  [06] Orig= 41 → Comp= 20 words (removed 21) | CoT signal=1.0 ✔verdict | Pred=1 | True=0 | ❌
  [07] Orig= 78 → Comp= 39 words (removed 39) | CoT signal=1.0 ✔verdict | Pred=1 | True=1 | ✅
  [08] Orig= 70 → Comp= 35 words (removed 35) | CoT signal=0.5 ⚠️ verdict LOST | Pred=1 | True=0 | ❌
  [09] Orig= 97 → Comp= 48 words (removed 49) | CoT signal=0.5 ⚠️ verdict LOST | Pred=1 | True=0 | ❌
  [10] Orig= 90 → Comp= 45 words (removed 45) | CoT signal=0.5 ⚠️ ver

,Tweet,True,Pred,✓?,CoT Signal,Verdict Survived,Orig Tokens,Comp Tokens,Tokens Removed,γ
0,@user Can U Help?||More conservatives ne...,0,1,❌,1.0,✔,87,43,44,0.5
1,Just walked in to #Starbucks and asked f...,1,1,✅,1.0,✔,81,40,41,0.5
2,#NOT GONNA WIN...,0,0,✅,0.0,✔,77,38,39,0.5
3,@user He is exactly that sort of person....,0,1,❌,0.5,✘ LOST,86,43,43,0.5
4,So much #sarcasm at work mate 10/10 #bor...,1,1,✅,1.0,✔,67,33,34,0.5
...,...,...,...,...,...,...,...,...,...,...
95,The last day of the year and I will be t...,0,1,❌,1.0,✔,89,44,45,0.5
96,MB forgot to move the elf on shelf and n...,0,1,❌,1.0,✔,84,42,42,0.5
97,"@user happy new year, da! Thanks sobra f...",0,1,❌,0.5,✘ LOST,59,29,30,0.5
98,WATCH: #Megatron from #Transformers Rant...,0,1,❌,0.5,✘ LOST,95,47,48,0.5
